# 08 - WEO Excel SciPy Statistics Lab

This lab uses the IMF World Economic Outlook Excel workbook from Module 4. You will create analysis-ready DataFrames from multiple sheets, then run descriptive statistics, confidence intervals, correlations, outlier checks, and hypothesis tests with SciPy.

## Important: Use a Python Notebook Runtime

Run this notebook with a **Python 3** kernel in Jupyter, VS Code, JupyterLab, or Google Colab. Do not run these cells in SQL Server query mode.

This notebook uses the WEO Excel workbook from Module 4. Locally, it looks for `Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx`. In Google Colab, upload `WEOApr2026all.xlsx` into the session files panel or use the upload prompt in the setup cell.

## 0. Setup

Run this setup cell in a Python notebook. It installs packages, finds the Excel workbook, and lists the workbook sheets.

In [ ]:
%pip install pandas numpy scipy matplotlib seaborn scikit-learn openpyxl joblib -q

from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def find_weo_workbook():
    """Find the WEO workbook whether the notebook runs from the lab folder, repo root, or Colab."""
    candidate_paths = [
        Path("../../../Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx"),
        Path("Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx"),
        Path("data/WEOApr2026all.xlsx"),
        Path("/content/WEOApr2026all.xlsx"),
        Path("WEOApr2026all.xlsx"),
    ]
    for path in candidate_paths:
        if path.exists():
            return path

    # Colab fallback: ask the learner to upload the Excel file if it is missing.
    try:
        from google.colab import files
        print("Upload WEOApr2026all.xlsx from Module 4 to continue.")
        uploaded = files.upload()
        if "WEOApr2026all.xlsx" in uploaded:
            return Path("WEOApr2026all.xlsx")
    except Exception:
        pass

    raise FileNotFoundError(
        "WEOApr2026all.xlsx was not found. Run locally from the repo, or upload the Excel file in Google Colab."
    )


WEO_PATH = find_weo_workbook()
print("Using workbook:", WEO_PATH)

excel_file = pd.ExcelFile(WEO_PATH)
print("Workbook sheets:", excel_file.sheet_names)

## 1. Create Relevant DataFrames from Multiple Sheets

The WEO file is wide: each year is a separate column. For analysis, we create:

- `country_long`: one row per country, indicator, and year.
- `country_macro`: one row per country and year with selected indicators as columns.
- `group_composition`: country-to-group mapping, used to compare advanced economies, emerging markets, and Sub-Saharan Africa.

In [ ]:
INDICATOR_LABELS = {
    "NGDP_RPCH": "gdp_growth_pct",
    "PCPIPCH": "inflation_pct",
    "LUR": "unemployment_rate",
    "BCA_NGDPD": "current_account_pct_gdp",
    "GGXWDG_NGDP": "government_debt_pct_gdp",
    "NGDPDPC": "gdp_per_capita_usd",
    "NID_NGDP": "investment_pct_gdp",
    "NGSD_NGDP": "savings_pct_gdp",
    "TX_RPCH": "export_volume_growth_pct",
    "TM_RPCH": "import_volume_growth_pct",
}


def get_year_columns(frame):
    """Return Excel columns that represent years such as 1980, 2024, or 2031."""
    return [column for column in frame.columns if isinstance(column, int) or str(column).isdigit()]


def load_weo_sheets(path):
    """Load the useful WEO workbook sheets into separate DataFrames."""
    countries = pd.read_excel(path, sheet_name="Countries")
    country_groups = pd.read_excel(path, sheet_name="Country Groups")
    commodity_prices = pd.read_excel(path, sheet_name="Commodity Prices")
    group_composition = pd.read_excel(path, sheet_name="Country Group Composition")
    return countries, country_groups, commodity_prices, group_composition


def make_long_indicator_data(frame, indicator_ids, source_name):
    """Convert WEO wide year columns into a tidy row-per-country-year format."""
    year_columns = get_year_columns(frame)
    id_columns = ["COUNTRY.ID", "COUNTRY", "INDICATOR.ID", "INDICATOR", "UNIT"]
    filtered = frame.loc[frame["INDICATOR.ID"].isin(indicator_ids), id_columns + year_columns].copy()

    long_df = filtered.melt(
        id_vars=id_columns,
        value_vars=year_columns,
        var_name="year",
        value_name="value",
    )
    long_df["year"] = long_df["year"].astype(int)
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df["source_sheet"] = source_name
    return long_df


def make_country_macro_dataframe(countries, group_composition):
    """Create one country-year row with selected WEO indicators as columns."""
    long_df = make_long_indicator_data(countries, list(INDICATOR_LABELS), "Countries")

    macro = (
        long_df.pivot_table(
            index=["COUNTRY.ID", "COUNTRY", "year"],
            columns="INDICATOR.ID",
            values="value",
            aggfunc="first",
        )
        .reset_index()
        .rename(columns=INDICATOR_LABELS)
    )

    advanced_codes = set(group_composition.loc[group_composition["groupname"].eq("Advanced Economies"), "countrycode"])
    emerging_codes = set(group_composition.loc[group_composition["groupname"].eq("Emerging Market and Developing Economies"), "countrycode"])
    ssa_codes = set(group_composition.loc[group_composition["groupname"].eq("Sub-Saharan Africa (SSA)"), "countrycode"])

    macro["economic_group"] = np.select(
        [macro["COUNTRY.ID"].isin(advanced_codes), macro["COUNTRY.ID"].isin(emerging_codes)],
        ["Advanced Economies", "Emerging Market and Developing Economies"],
        default="Other",
    )
    macro["is_sub_saharan_africa"] = macro["COUNTRY.ID"].isin(ssa_codes)
    return macro, long_df


countries_raw, country_groups_raw, commodity_prices_raw, group_composition = load_weo_sheets(WEO_PATH)
country_macro, country_long = make_country_macro_dataframe(countries_raw, group_composition)

print("Countries sheet rows:", len(countries_raw))
print("Country-year analytical rows:", len(country_macro))
display(country_macro.head())

## 2. EDA Summary for a Recent Year

Start with a recent forecast year. The April 2026 WEO file includes actual history and forecast years, so treat recent years as WEO estimates/projections.

In [ ]:
analysis_year = 2026
latest = country_macro.loc[country_macro["year"].eq(analysis_year)].copy()

selected_columns = [
    "COUNTRY",
    "economic_group",
    "is_sub_saharan_africa",
    "gdp_growth_pct",
    "inflation_pct",
    "unemployment_rate",
    "government_debt_pct_gdp",
    "current_account_pct_gdp",
    "gdp_per_capita_usd",
]

display(latest[selected_columns].head(10))
display(latest[selected_columns[3:]].describe().T.round(2))

## 3. SciPy Descriptive Statistics

`stats.describe()` gives count, min/max, mean, variance, skewness, and kurtosis for one numeric variable.

In [ ]:
from scipy import stats

growth = latest["gdp_growth_pct"].dropna().to_numpy()
growth_stats = stats.describe(growth)

print("Number of countries:", growth_stats.nobs)
print("Minimum and maximum GDP growth:", tuple(round(value, 2) for value in growth_stats.minmax))
print("Mean GDP growth:", round(growth_stats.mean, 2))
print("Variance:", round(growth_stats.variance, 2))
print("Skewness:", round(growth_stats.skewness, 2))
print("Kurtosis:", round(growth_stats.kurtosis, 2))

## 4. Confidence Interval for Average GDP Growth

This estimates a range for the average country-level real GDP growth rate in the selected year.

In [ ]:
mean_growth = np.mean(growth)
standard_error = stats.sem(growth)
confidence_interval = stats.t.interval(
    confidence=0.95,
    df=len(growth) - 1,
    loc=mean_growth,
    scale=standard_error,
)

print(f"{analysis_year} mean GDP growth:", round(mean_growth, 2))
print("95% confidence interval:", tuple(round(value, 2) for value in confidence_interval))

## 5. Correlation Scenario: Inflation and GDP Growth

Business question: in the recent WEO projection year, do countries with higher inflation also show different GDP growth patterns?

In [ ]:
corr_df = latest[["gdp_growth_pct", "inflation_pct"]].dropna()
pearson_corr, pearson_p = stats.pearsonr(corr_df["inflation_pct"], corr_df["gdp_growth_pct"])
spearman_corr, spearman_p = stats.spearmanr(corr_df["inflation_pct"], corr_df["gdp_growth_pct"])

print("Pearson correlation:", round(pearson_corr, 3), "p-value:", round(pearson_p, 4))
print("Spearman correlation:", round(spearman_corr, 3), "p-value:", round(spearman_p, 4))

## 6. Outlier Review with Z-Scores

A z-score is useful for quickly finding countries with unusually high or low values compared with the rest of the dataset.

In [ ]:
latest["growth_z_score"] = stats.zscore(latest["gdp_growth_pct"], nan_policy="omit")
latest["growth_outlier_flag"] = latest["growth_z_score"].abs() > 2

outlier_columns = ["COUNTRY", "economic_group", "gdp_growth_pct", "growth_z_score"]
display(latest.loc[latest["growth_outlier_flag"], outlier_columns].sort_values("growth_z_score"))

## 7. Hypothesis Test Scenario

Question: is average projected GDP growth different between advanced economies and emerging market/developing economies?

In [ ]:
advanced_growth = latest.loc[latest["economic_group"].eq("Advanced Economies"), "gdp_growth_pct"].dropna()
emerging_growth = latest.loc[latest["economic_group"].eq("Emerging Market and Developing Economies"), "gdp_growth_pct"].dropna()

t_stat, p_value = stats.ttest_ind(advanced_growth, emerging_growth, equal_var=False)

print("Advanced economies mean:", round(advanced_growth.mean(), 2))
print("Emerging/developing economies mean:", round(emerging_growth.mean(), 2))
print("t-statistic:", round(t_stat, 3))
print("p-value:", round(p_value, 4))

if p_value < 0.05:
    print("Interpretation: the group difference is statistically significant at the 5% level.")
else:
    print("Interpretation: the group difference is not statistically significant at the 5% level.")

## 8. Export Statistical Outputs

In [ ]:
summary_path = OUTPUT_DIR / "weo_scipy_2026_summary.csv"
outliers_path = OUTPUT_DIR / "weo_scipy_growth_outliers.csv"

latest[selected_columns[3:]].describe().T.round(2).to_csv(summary_path)
latest.loc[latest["growth_outlier_flag"], outlier_columns].sort_values("growth_z_score").to_csv(outliers_path, index=False)

print("Saved:", summary_path)
print("Saved:", outliers_path)